# 🔍 Milestone 3 — Retrieval-Augmented Generation (RAG)

**Roll No**: 24f1002384 | **Course**: DL/Gen AI  

This notebook answers all Milestone 3 questions on:  
- FAISS vector retrieval  
- Cross-Encoder reranking  
- RAG-augmented zero-shot classification  
- Hit Rate evaluation  
- Adversarial RAG  
- Full RAG pipeline MAP@3  

---

## ⚙️ Setup (Run First)

In [ ]:
import subprocess, sys
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q',
     'faiss-cpu', 'sentence-transformers', 'transformers', 'torch', 'pandas', 'numpy'],
    check=True
)
print('Packages ready')

In [ ]:
# === MILESTONE 3 SETUP (as specified in the assignment) ===
import pandas as pd
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

DATA_DIR = '/kaggle/input/competitions/smart-mcq-solver-challenge'
train = pd.read_csv(f'{DATA_DIR}/train.csv')

CHOICES = list('ABCDE')
LABEL2IDX = {c: i for i, c in enumerate(CHOICES)}
IDX2LABEL = {i: c for c, i in LABEL2IDX.items()}

print('Creating knowledge base...')
kb = []
for idx, row in train.iterrows():
    correct_letter = row['answer']
    kb.append(str(row[correct_letter]))

print('Loading embedding model and creating FAISS index...')
model = SentenceTransformer('all-MiniLM-L6-v2')
kb_embeddings = model.encode(kb, show_progress_bar=True)
kb_embeddings = kb_embeddings.astype('float32')

index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)

print(f'Knowledge base successfully created: {len(kb)} documents, embedding dim={kb_embeddings.shape[1]}')

In [ ]:
# Zero-shot classifier setup (for Q1, Q2, Q6)
print('Loading facebook/bart-large-mnli zero-shot classifier...')
zs = pipeline('zero-shot-classification', model='facebook/bart-large-mnli', device=-1)

row_150 = train.iloc[150]
prompt_150 = str(row_150['prompt'])
labels_150 = [str(row_150['A']), str(row_150['B']), str(row_150['C']),
               str(row_150['D']), str(row_150['E'])]
ans_150 = str(row_150[row_150['answer']])

print(f'Row 150 prompt: {prompt_150[:80]}...')
print(f'Row 150 correct answer ({row_150["answer"]}): {ans_150[:60]}...')

## ❓ Q1 — Zero-Shot Classification on Row 150

Run the zero-shot classifier on `facebook/bart-large-mnli` on prompt for row index 150.  
Pass the 5 options (A-E) as `candidate_labels`.  
**Report the predicted probability score assigned to the ground-truth correct option (rounded to 3 decimal places).**

In [ ]:
# Q1: Zero-shot on row 150 with 5 options as candidate_labels
zs_result_150 = zs(prompt_150, candidate_labels=labels_150)

# Map scores back to option letters
label_to_score = dict(zip(zs_result_150['labels'], zs_result_150['scores']))

# Find score for the correct option
correct_score_q1 = label_to_score[ans_150]

print(f'All scores:')
for lbl, sc in zip(zs_result_150['labels'], zs_result_150['scores']):
    marker = ' <-- CORRECT' if lbl == ans_150 else ''
    print(f'  {lbl[:50]:50s}  {sc:.4f}{marker}')

print(f'\n>>> Q1 ANSWER: {correct_score_q1:.3f}')

## ❓ Q2 — FAISS Top-10 Retrieval Rank of True Document

Embed the prompt for row 150 using `all-MiniLM-L6-v2`.  
Query FAISS index for top k=10. At what exact rank (1–10) did FAISS place the true correct document?

In [ ]:
# Q2: Embed prompt_150 and retrieve top 10 from FAISS
q_emb_150 = model.encode([prompt_150]).astype('float32')  # (1, dim)
distances, retrieved_indices = index.search(q_emb_150, 10)  # top 10

retrieved_indices_flat = retrieved_indices[0].tolist()

print('Top 10 retrieved KB indices (rank 1 → 10):')
rank_of_true = None
for rank, idx in enumerate(retrieved_indices_flat, 1):
    marker = ' <-- TRUE DOCUMENT (kb[150])' if idx == 150 else ''
    print(f'  Rank {rank}: kb[{idx}]  dist={distances[0][rank-1]:.4f}{marker}')
    if idx == 150:
        rank_of_true = rank

if rank_of_true is None:
    print('\nTrue document NOT in top 10!')
    rank_of_true = 'NOT IN TOP 10'

print(f'\n>>> Q2 ANSWER: Rank {rank_of_true}')

## ❓ Q3 — Cross-Encoder Reranking

Take the top 10 documents from Q2. Load `cross-encoder/ms-marco-MiniLM-L-6-v2`.  
Score prompt vs each of the 10 documents. At what rank does the cross-encoder place the true document?

In [ ]:
# Q3: Cross-encoder reranking of top-10 FAISS results
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

docs_10 = [kb[i] for i in retrieved_indices_flat]  # Top 10 documents
pairs = [[prompt_150, doc] for doc in docs_10]
ce_scores = cross_encoder.predict(pairs)

# Sort by cross-encoder score descending
ce_ranked = sorted(
    zip(retrieved_indices_flat, docs_10, ce_scores),
    key=lambda x: -x[2]
)

print('Cross-encoder reranked order:')
ce_rank_of_true = None
for rank, (kb_idx, doc, score) in enumerate(ce_ranked, 1):
    marker = ' <-- TRUE DOCUMENT' if kb_idx == 150 else ''
    print(f'  Rank {rank}: kb[{kb_idx}]  CE_score={score:.4f}{marker}')
    if kb_idx == 150:
        ce_rank_of_true = rank

if ce_rank_of_true is None:
    ce_rank_of_true = 'NOT IN TOP 10'

print(f'\n>>> Q3 ANSWER: Rank {ce_rank_of_true}')

## ❓ Q4 — Token Count for RAG String (row 42, k=5)

Retrieve top k=5 for row index 42. Concatenate with a space.  
Create: `"Context: [docs] Question: [prompt]"`. Tokenize with `bert-base-uncased` (no truncation).  
**How many total tokens?**

In [ ]:
# Q4: Token count for row 42 RAG string
row_42 = train.iloc[42]
prompt_42 = str(row_42['prompt'])

# Embed prompt_42 and retrieve top 5
q_emb_42 = model.encode([prompt_42]).astype('float32')
dist_42, idx_42 = index.search(q_emb_42, 5)
docs_5_42 = [kb[i] for i in idx_42[0]]

# Concatenate docs with single space
concatenated_docs = ' '.join(docs_5_42)

# Build RAG string
rag_string_42 = f'Context: {concatenated_docs} Question: {prompt_42}'

print(f'RAG string (first 200 chars): {rag_string_42[:200]}...')
print(f'RAG string length: {len(rag_string_42)} characters')

# Tokenize with bert-base-uncased (no truncation)
bert_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
tokens = bert_tokenizer(rag_string_42, truncation=False)
total_tokens = len(tokens['input_ids'])

print(f'\n>>> Q4 ANSWER: {total_tokens} total tokens')

## ❓ Q5 — RAG-Augmented Zero-Shot on Row 150

Retrieve the **exact true document** for row 150 from KB (i.e., `kb[150]`).  
Create: `"Context: [true_document] Question: [prompt]"`.  
Run zero-shot classification. **What is the new probability of the correct option?**

In [ ]:
# Q5: RAG-augmented zero-shot with the TRUE document for row 150
true_doc_150 = kb[150]  # The exact correct answer text

rag_string_150 = f'Context: {true_doc_150} Question: {prompt_150}'

print(f'True document: {true_doc_150[:100]}...')
print(f'\nRAG string (first 200 chars): {rag_string_150[:200]}...')

# Zero-shot classification on the augmented string
zs_rag_result = zs(rag_string_150, candidate_labels=labels_150)

label_to_score_rag = dict(zip(zs_rag_result['labels'], zs_rag_result['scores']))
correct_score_q5 = label_to_score_rag[ans_150]

print(f'\nAll RAG-augmented scores:')
for lbl, sc in zip(zs_rag_result['labels'], zs_rag_result['scores']):
    marker = ' <-- CORRECT' if lbl == ans_150 else ''
    print(f'  {lbl[:50]:50s}  {sc:.4f}{marker}')

print(f'\nQ1 score (no context): {correct_score_q1:.3f}')
print(f'>>> Q5 ANSWER (with true context): {correct_score_q5:.3f}')

## ❓ Q6 — Adversarial RAG (Wrong Document)

Force context to be **`kb[999]`** (completely unrelated).  
Run zero-shot on this adversarial RAG string.  
**What is the probability of the correct option now?**

In [ ]:
# Q6: Adversarial RAG — inject wrong document kb[999]
adversarial_doc = kb[999]

adversarial_rag_string = f'Context: {adversarial_doc} Question: {prompt_150}'

print(f'Adversarial document (kb[999]): {adversarial_doc[:100]}...')
print(f'\nAdversarial RAG string (first 200 chars): {adversarial_rag_string[:200]}...')

zs_adv_result = zs(adversarial_rag_string, candidate_labels=labels_150)

label_to_score_adv = dict(zip(zs_adv_result['labels'], zs_adv_result['scores']))
correct_score_q6 = label_to_score_adv[ans_150]

print(f'\nAll adversarial RAG scores:')
for lbl, sc in zip(zs_adv_result['labels'], zs_adv_result['scores']):
    marker = ' <-- CORRECT' if lbl == ans_150 else ''
    print(f'  {lbl[:50]:50s}  {sc:.4f}{marker}')

print(f'\nQ1 (no context):       {correct_score_q1:.3f}')
print(f'Q5 (true context):     {correct_score_q5:.3f}')
print(f'>>> Q6 ANSWER (wrong context): {correct_score_q6:.3f}')

## ❓ Q7 — Hit Rate for First 100 Rows

For rows 0–99, retrieve top k=5 documents.  
A **hit** = the exact correct option string is found inside any of the 5 retrieved docs.  
**What is the Hit Rate % (0–100, rounded to 1 decimal)?**

In [ ]:
# Q7: Hit Rate for first 100 rows
hits = 0
EVAL_ROWS = 100

for i in range(EVAL_ROWS):
    row = train.iloc[i]
    prompt_i = str(row['prompt'])
    correct_text = str(row[row['answer']])  # The exact correct option text

    # Embed and retrieve top 5
    q_emb = model.encode([prompt_i]).astype('float32')
    _, idx_i = index.search(q_emb, 5)
    retrieved_docs = [kb[j] for j in idx_i[0]]

    # Check if exact correct text is in any retrieved document
    is_hit = any(correct_text in doc for doc in retrieved_docs)
    if is_hit:
        hits += 1

hit_rate = hits / EVAL_ROWS * 100
print(f'Hits: {hits} / {EVAL_ROWS}')
print(f'>>> Q7 ANSWER: {hit_rate:.1f}%')

## ❓ Q8 — Full RAG Pipeline MAP@3 for First 20 Rows

For rows 0–19:  
1. Retrieve top k=5 from FAISS  
2. Cross-encode → pick best document  
3. Build `"Context: [best_doc] Question: [prompt]"`  
4. Zero-shot classify with 5 options  
5. Rank by probability → MAP@3  

**What is the average MAP@3 across 20 rows?**

In [ ]:
# Q8: Full RAG pipeline MAP@3 for first 20 rows
def map_at_3_single(pred_top3: list, true_label: str) -> float:
    for rank, opt in enumerate(pred_top3[:3], 1):
        if opt == true_label:
            return 1.0 / rank
    return 0.0

all_maps = []
PIPELINE_ROWS = 20

for i in range(PIPELINE_ROWS):
    row = train.iloc[i]
    prompt_i = str(row['prompt'])
    true_label = row['answer']
    option_texts = [str(row[c]) for c in CHOICES]

    # Step 1: Retrieve top k=5 from FAISS
    q_emb = model.encode([prompt_i]).astype('float32')
    _, idx_i = index.search(q_emb, 5)
    retrieved_docs = [kb[j] for j in idx_i[0]]

    # Step 2: Cross-encoder reranking → pick best
    pairs_i = [[prompt_i, doc] for doc in retrieved_docs]
    ce_scores_i = cross_encoder.predict(pairs_i)
    best_doc = retrieved_docs[int(np.argmax(ce_scores_i))]

    # Step 3: Augment prompt
    rag_str = f'Context: {best_doc} Question: {prompt_i}'

    # Step 4: Zero-shot classify
    result = zs(rag_str, candidate_labels=option_texts)

    # Step 5: Rank options by probability, map back to A-E letters
    label_to_letter = {str(row[c]): c for c in CHOICES}
    ranked_letters = []
    for lbl in result['labels']:  # Already sorted high→low
        letter = label_to_letter.get(lbl)
        if letter:
            ranked_letters.append(letter)

    # MAP@3 for this row
    score = map_at_3_single(ranked_letters, true_label)
    all_maps.append(score)
    print(f'Row {i:2d} | True={true_label} | Top3={ranked_letters[:3]} | AP={score:.3f}')

final_map3 = np.mean(all_maps)
print(f'\n>>> Q8 ANSWER: {final_map3:.3f}')

In [ ]:
# ===== ANSWERS SUMMARY =====
print('=' * 50)
print('MILESTONE 3 ANSWERS SUMMARY')
print('=' * 50)
print(f'Q1: {correct_score_q1:.3f}')
print(f'Q2: Rank {rank_of_true}')
print(f'Q3: Rank {ce_rank_of_true}')
print(f'Q4: {total_tokens} tokens')
print(f'Q5: {correct_score_q5:.3f}')
print(f'Q6: {correct_score_q6:.3f}')
print(f'Q7: {hit_rate:.1f}%')
print(f'Q8: {final_map3:.3f}')
print('=' * 50)